<a id='9c-13'></a>

## 9c-13. 🧩 Pattern 13: collections module — LC 49, 347, 146, 239

---

```
PROBLEM:  Standard dict/list fall short for counting, ordering, defaulting,
          and fixed-size sliding windows. collections fills those gaps.

APPROACH: Pick the right container for the job:
  Counter      — frequency map in one line
  defaultdict  — auto-creates missing keys, no KeyError
  OrderedDict  — insertion-order dict (LRU cache building block)
  deque        — O(1) both-end push/pop, bounded window with maxlen
  namedtuple   — lightweight struct with field names instead of indices

SLOW MOTION TRACE:
  # Counter
  Counter("aabb")  ->  {'a':2, 'b':2}
  Counter([1,2,2]) ->  {2:2, 1:1}
  c.most_common(2) ->  [(most_freq, count), (second, count)]

  # defaultdict(list) — group-by pattern
  d = defaultdict(list)
  d['key'].append(1)   # no KeyError — key auto-created with []
  d['key'].append(2)   # d == {'key': [1, 2]}

  # deque(maxlen=3) — sliding window eviction
  dq = deque(maxlen=3)
  dq.extend([1,2,3])   -> deque([1,2,3])
  dq.append(4)         -> deque([2,3,4])   leftmost evicted automatically

KEY INSIGHT: Counter + most_common() replaces sort-on-frequency in O(n);
             defaultdict eliminates if-key-in-dict boilerplate everywhere.

TIME / SPACE:
  Counter(iterable)  O(n) time, O(k) space (k=unique elements)
  defaultdict ops    O(1) per access
  deque append/pop   O(1) both ends
```


In [ ]:
from collections import Counter, defaultdict, OrderedDict, deque, namedtuple
from typing import List


# ── Counter — frequency map in one line ──────────────────────────────────────
words = ["apple", "banana", "apple", "cherry", "banana", "apple"]
freq = Counter(words)
print(freq)                         # Counter({'apple': 3, 'banana': 2, 'cherry': 1})
print(freq.most_common(2))          # [('apple', 3), ('banana', 2)]
print(freq["apple"])                # 3
print(freq["mango"])                # 0  — missing keys return 0, no KeyError

# Arithmetic on counters
c1 = Counter("aab")
c2 = Counter("ab")
print(c1 - c2)                      # Counter({'a': 1})  — subtract counts


# ── LC 49 — Group Anagrams using defaultdict ──────────────────────────────────
def group_anagrams(strs: List[str]) -> List[List[str]]:
    """
    LC 49 — Group Anagrams.
    Approach: sorted word as key, defaultdict(list) for grouping.
    Args:
        strs (List[str]): list of strings.
    Returns:
        List[List[str]]: groups of anagrams.
    Time:  O(n * k log k) — n words, k = max word length
    Space: O(n * k) — storing all words in groups
    """
    groups: defaultdict = defaultdict(list)   # auto-creates [] for new keys
    for word in strs:
        key = "".join(sorted(word))           # sorted letters = canonical anagram key
        groups[key].append(word)              # no KeyError — defaultdict handles it
    return list(groups.values())

def test_harness_anagrams(fn):
    tests = [
        (["eat","tea","tan","ate","nat","bat"],
         [["eat","tea","ate"],["tan","nat"],["bat"]]),
        ([""], [[""]]),
        (["a"], [["a"]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        got_sorted   = sorted(sorted(g) for g in got)
        exp_sorted   = sorted(sorted(g) for g in expected)
        status = "PASSED" if got_sorted == exp_sorted else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got_sorted == exp_sorted)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_anagrams(group_anagrams)
print("group_anagrams defined.")


# ── LC 347 — Top K Frequent using Counter ────────────────────────────────────
def top_k_frequent(nums: List[int], k: int) -> List[int]:
    """
    LC 347 — Top K Frequent Elements.
    Approach: Counter + most_common(k).
    Args:
        nums (List[int]): input list.
        k (int): number of top frequent elements.
    Returns:
        List[int]: k most frequent elements.
    Time:  O(n log k) — Counter O(n), most_common O(n log k)
    Space: O(n) — counter stores all unique elements
    """
    return [val for val, _ in Counter(nums).most_common(k)]  # unpack (val, count) tuples

def test_harness_topk(fn):
    tests = [
        ([1,1,1,2,2,3], 2, [1, 2]),
        ([1], 1, [1]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if sorted(got) == sorted(expected) else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (sorted(got) == sorted(expected))
    print(f"{passed}/{len(tests)} tests passed")

test_harness_topk(top_k_frequent)
print("top_k_frequent defined.")


# ── deque — O(1) both-end operations ─────────────────────────────────────────
dq = deque([1, 2, 3])
dq.appendleft(0)            # O(1) — push to front
dq.append(4)                # O(1) — push to back
print(dq)                   # deque([0, 1, 2, 3, 4])
print(dq.popleft())         # 0  — O(1) pop from front
print(dq.pop())             # 4  — O(1) pop from back

# maxlen: automatic eviction — sliding window
window = deque(maxlen=3)
for x in [1, 2, 3, 4, 5]:
    window.append(x)
    print(f"  window={list(window)}")   # oldest element auto-evicted when full


# ── namedtuple — struct without class boilerplate ────────────────────────────
Point  = namedtuple("Point",  ["x", "y"])
Record = namedtuple("Record", ["name", "score", "rank"])

p = Point(3, 7)
r = Record("Sean", 98, 1)
print(f"point: x={p.x}, y={p.y}")         # field names instead of [0],[1]
print(f"record: {r.name} scored {r.score}")

# Simplicity and clarity is Gold
